In [36]:
import torch
import torch.nn as nn
class LinkNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv2d1=nn.Conv2d(in_channels=3,out_channels=64,kernel_size=7,stride=2,padding=3)
        self.pool = nn.MaxPool2d(kernel_size=3, stride=2)
        self.enc1 = encoder_block(64, 64)
        self.enc2 = encoder_block(64, 128)
        self.enc3 = encoder_block(128, 256)
        self.enc4 = encoder_block(256, 512)

        self.dec4 = decoder_block(512, 256)
        self.dec3 = decoder_block(256, 128)
        self.dec2 = decoder_block(128, 64)
        self.dec1 = decoder_block(64, 64)
        self.up_train = nn.ConvTranspose2d(in_channels=64,out_channels=32,kernel_size=3,stride=2,padding=1, output_padding=1)
        self.conv2d2=nn.Conv2d(in_channels=32,out_channels=32,kernel_size=3,padding=1)
        self.up_train1 = nn.ConvTranspose2d(in_channels=32,out_channels=3,kernel_size=2,stride=2)
       

    def forward(self, img):
        x=self.conv2d1(img)
        x=self.pool(x)
        x, skip1 = self.enc1(x)
        x, skip2 = self.enc2(x)
        x, skip3 = self.enc3(x)
        x, skip4 = self.enc4(x)
        x = self.dec4(x)+skip3
        x = self.dec3(x) + skip2
        x = self.dec2(x) + skip1
        x = self.dec1(x) 
        print(x.shape)
        x=self.up_train(x)
        x=self.conv2d2(x)
        x=self.up_train1(x)
        
        return x
class encoder_block(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.input_dim=input_dim
        self.output_dim=output_dim
        self.conv2d1=nn.Conv2d(in_channels=self.input_dim,out_channels=self.output_dim,kernel_size=3,stride=2,padding=1)
        self.conv2d2=nn.Conv2d(in_channels=self.output_dim,out_channels=self.output_dim,kernel_size=3,padding=1)
        self.conv2d3=nn.Conv2d(in_channels=self.output_dim,out_channels=self.output_dim,kernel_size=3,padding=1)
        self.conv2d4=nn.Conv2d(in_channels=self.output_dim,out_channels=self.output_dim,kernel_size=3,padding=1)
        self.shortcut=nn.Conv2d(in_channels=self.input_dim,out_channels=self.output_dim,kernel_size=3,stride=2,padding=1)
        #self.shortcut1=nn.Conv2d(in_channels=self.output_dim,out_channels=self.output_dim,kernel_size=3,padding=1)
    def forward(self,x):
        x1=self.conv2d1(x)
        short=self.shortcut(x)
        y=self.conv2d2(x1)
        res=short+y
        x=self.conv2d3(res)
        x=self.conv2d4(x)
        out=res+x
        skip=out
        return out,skip
class decoder_block(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.input_dim=input_dim
        self.output_dim=output_dim
        self.conv2d1=nn.Conv2d(in_channels=self.input_dim,out_channels=self.input_dim//4,kernel_size=1)
        self.up_train = nn.ConvTranspose2d(in_channels=self.input_dim//4,out_channels=self.input_dim//4,kernel_size=3,stride=2,padding=1, output_padding=1)
        self.conv2d2=nn.Conv2d(in_channels=self.input_dim//4,out_channels=self.output_dim,kernel_size=1)
    def forward(self,x):
        x=self.conv2d1(x)
        x=self.up_train(x)
        x=self.conv2d2(x)
        return x

In [37]:
model = LinkNet()
    # 构造模拟图片：batch=2，3通道，高256，宽512
fake_img = torch.randn(2, 3, 256, 512)
output = model(fake_img)
print("输入尺寸：", fake_img.shape)
print("输出尺寸：", output.shape)



torch.Size([2, 64, 64, 128])
输入尺寸： torch.Size([2, 3, 256, 512])
输出尺寸： torch.Size([2, 3, 256, 512])
